# 🚀 POC 2: CD Pipeline — Deploy ML Model to Kubernetes

**What you'll learn:**
- Kubernetes manifests (Deployment, Service, HPA)
- Harness Service, Environment, Infrastructure definitions
- Rolling, Canary, Blue-Green deployment pipelines
- Approval gates & automatic rollback

```
Docker Image → Deploy Staging → Verify → Approve → Deploy Prod
```

## Step 1: Kubernetes Manifests

These YAML files tell Kubernetes HOW to run your app:
- **Deployment** → How many pods, what image, health checks
- **Service** → Network endpoint to reach your pods
- **HPA** → Auto-scale pods based on CPU/memory

In [ ]:
# Step 1: Generate Kubernetes manifests with Python
import yaml, os

# ---- Deployment Manifest ----
deployment = {
    "apiVersion": "apps/v1",
    "kind": "Deployment",
    "metadata": {
        "name": "ml-sentiment-api",
        "namespace": "ml-platform",
        "labels": {"app": "ml-sentiment-api"}
    },
    "spec": {
        "replicas": 3,
        "selector": {"matchLabels": {"app": "ml-sentiment-api"}},
        "strategy": {
            "type": "RollingUpdate",
            "rollingUpdate": {"maxSurge": 1, "maxUnavailable": 0}
        },
        "template": {
            "metadata": {"labels": {"app": "ml-sentiment-api"}},
            "spec": {
                "containers": [{
                    "name": "api",
                    "image": "myuser/ml-sentiment-api:latest",  # Harness replaces this
                    "ports": [{"containerPort": 8000}],
                    "resources": {
                        "requests": {"cpu": "100m", "memory": "256Mi"},
                        "limits": {"cpu": "500m", "memory": "512Mi"}
                    },
                    # Liveness: Is app alive? If no → RESTART pod
                    "livenessProbe": {
                        "httpGet": {"path": "/health", "port": 8000},
                        "initialDelaySeconds": 15,
                        "periodSeconds": 10
                    },
                    # Readiness: Is app ready? If no → STOP sending traffic
                    "readinessProbe": {
                        "httpGet": {"path": "/health/ready", "port": 8000},
                        "initialDelaySeconds": 5,
                        "periodSeconds": 5
                    }
                }]
            }
        }
    }
}

# ---- Service Manifest ----
service = {
    "apiVersion": "v1",
    "kind": "Service",
    "metadata": {"name": "ml-sentiment-api", "namespace": "ml-platform"},
    "spec": {
        "type": "ClusterIP",
        "selector": {"app": "ml-sentiment-api"},
        "ports": [{"port": 8000, "targetPort": 8000}]
    }
}

# Save manifests
os.makedirs('k8s', exist_ok=True)
for name, manifest in [("deployment", deployment), ("service", service)]:
    with open(f'k8s/{name}.yaml', 'w') as f:
        yaml.dump(manifest, f, default_flow_style=False, sort_keys=False)
    print(f"✅ Created k8s/{name}.yaml")

print("\n📋 Deployment YAML:")
print(yaml.dump(deployment, default_flow_style=False, sort_keys=False)[:500] + "...")

: 

## Step 2: Harness Service & Environment

| Concept | What | Example |
|---------|------|---------|
| **Service** | WHAT you deploy | Docker image + K8s manifests |
| **Environment** | WHERE you deploy | staging, production |
| **Infrastructure** | Specific cluster details | EKS cluster in us-east-1 |

In [ ]:
# Step 2: Generate Harness Service + Environment YAML

harness_service = {
    "service": {
        "name": "ML Sentiment API",
        "identifier": "ml_sentiment_api",
        "serviceDefinition": {
            "type": "Kubernetes",
            "spec": {
                "manifests": [{
                    "manifest": {
                        "identifier": "k8s_manifests",
                        "type": "K8sManifest",
                        "spec": {
                            "store": {
                                "type": "Github",
                                "spec": {
                                    "connectorRef": "github_connector",
                                    "paths": ["k8s/"]
                                }
                            }
                        }
                    }
                }],
                "artifacts": {
                    "primary": {
                        "sources": [{
                            "identifier": "docker_image",
                            "type": "DockerRegistry",
                            "spec": {
                                "connectorRef": "dockerhub_connector",
                                "imagePath": "myuser/ml-sentiment-api",
                                "tag": "<+input>"
                            }
                        }]
                    }
                }
            }
        }
    }
}

staging_env = {
    "environment": {
        "name": "Staging",
        "identifier": "staging",
        "type": "PreProduction",
        "variables": [
            {"name": "replicas", "value": "2"},
            {"name": "log_level", "value": "debug"}
        ]
    }
}

prod_env = {
    "environment": {
        "name": "Production",
        "identifier": "production",
        "type": "Production",
        "variables": [
            {"name": "replicas", "value": "5"},
            {"name": "log_level", "value": "warn"}
        ]
    }
}

os.makedirs('.harness', exist_ok=True)
for name, config in [("service", harness_service), ("env_staging", staging_env), ("env_prod", prod_env)]:
    with open(f'.harness/{name}.yaml', 'w') as f:
        yaml.dump(config, f, default_flow_style=False, sort_keys=False)
    print(f"✅ Created .harness/{name}.yaml")

## Step 3: Three Deployment Strategies

We'll generate all 3 pipeline YAMLs so you can compare them.

In [ ]:
# Step 3A: Rolling Deployment Pipeline

def make_deploy_stage(name, env, strategy_steps, rollback_step_type):
    """Helper to build a deployment stage"""
    return {
        "stage": {
            "name": name,
            "type": "Deployment",
            "spec": {
                "deploymentType": "Kubernetes",
                "service": {"serviceRef": "ml_sentiment_api"},
                "environment": {
                    "environmentRef": env,
                    "infrastructureDefinitions": [{"identifier": f"{env}_k8s"}]
                },
                "execution": {
                    "steps": strategy_steps,
                    "rollbackSteps": [{
                        "step": {
                            "type": rollback_step_type,
                            "name": "Rollback",
                            "spec": {}
                        }
                    }]
                }
            }
        }
    }

def make_approval_stage():
    return {
        "stage": {
            "name": "Approval Gate",
            "type": "Approval",
            "spec": {
                "execution": {
                    "steps": [{
                        "step": {
                            "type": "HarnessApproval",
                            "name": "Approve Production",
                            "spec": {
                                "approvalMessage": "Approve production deployment?",
                                "approvers": {"userGroups": ["_project_all_users"], "minimumCount": 1}
                            },
                            "timeout": "1d"
                        }
                    }]
                }
            }
        }
    }

# ---- ROLLING DEPLOY PIPELINE ----
rolling_steps = [
    {"step": {"type": "K8sRollingDeploy", "name": "Rolling Deploy", "spec": {"skipDryRun": False}, "timeout": "10m"}}
]

rolling_pipeline = {
    "pipeline": {
        "name": "CD - Rolling Deployment",
        "identifier": "cd_rolling",
        "stages": [
            make_deploy_stage("Deploy Staging", "staging", rolling_steps, "K8sRollingRollback"),
            make_approval_stage(),
            make_deploy_stage("Deploy Production", "production", rolling_steps, "K8sRollingRollback"),
        ]
    }
}

with open('.harness/cd-rolling.yaml', 'w') as f:
    yaml.dump(rolling_pipeline, f, default_flow_style=False, sort_keys=False)
print("✅ Created .harness/cd-rolling.yaml")
print("\n🔄 Rolling: Staging → Approve → Production")

In [ ]:
# Step 3B: Canary Deployment Pipeline

canary_steps = [
    {"step": {"type": "K8sCanaryDeploy", "name": "Canary 20%",
              "spec": {"instanceSelection": {"type": "Percentage", "spec": {"percentage": 20}}},
              "timeout": "10m"}},
    {"step": {"type": "ShellScript", "name": "Verify Canary",
              "spec": {"shell": "Bash", "source": {"type": "Inline",
                       "spec": {"script": "echo 'Verifying canary health...' && sleep 30 && echo 'Healthy!'"}}}}},
    {"step": {"type": "HarnessApproval", "name": "Approve Full Rollout",
              "spec": {"approvalMessage": "Canary is healthy. Approve 100% rollout?",
                       "approvers": {"userGroups": ["_project_all_users"], "minimumCount": 1}}}},
    {"step": {"type": "K8sRollingDeploy", "name": "Full Rollout 100%",
              "spec": {"skipDryRun": False}}},
    {"step": {"type": "K8sCanaryDelete", "name": "Cleanup Canary", "spec": {}}}
]

canary_pipeline = {
    "pipeline": {
        "name": "CD - Canary Deployment",
        "identifier": "cd_canary",
        "stages": [
            make_deploy_stage("Canary to Production", "production", canary_steps, "K8sCanaryDelete")
        ]
    }
}

with open('.harness/cd-canary.yaml', 'w') as f:
    yaml.dump(canary_pipeline, f, default_flow_style=False, sort_keys=False)
print("✅ Created .harness/cd-canary.yaml")
print("\n🐤 Canary: Deploy 20% → Verify → Approve → Rollout 100%")

In [ ]:
# Step 3C: Blue-Green Deployment Pipeline

bluegreen_steps = [
    {"step": {"type": "K8sBlueGreenDeploy", "name": "Stage Green",
              "spec": {"skipDryRun": False}, "timeout": "15m"}},
    {"step": {"type": "ShellScript", "name": "Verify Green",
              "spec": {"shell": "Bash", "source": {"type": "Inline",
                       "spec": {"script": "echo 'Green deployed, not serving traffic yet...' && sleep 10"}}}}},
    {"step": {"type": "HarnessApproval", "name": "Approve Swap",
              "spec": {"approvalMessage": "Swap traffic Blue→Green?",
                       "approvers": {"userGroups": ["_project_all_users"], "minimumCount": 1}}}},
    {"step": {"type": "K8sBGSwapServices", "name": "Swap Traffic",
              "spec": {"skipDryRun": False}}}
]

bg_pipeline = {
    "pipeline": {
        "name": "CD - Blue-Green Deployment",
        "identifier": "cd_bluegreen",
        "stages": [
            make_deploy_stage("Blue-Green Deploy", "production", bluegreen_steps, "K8sBGSwapServices")
        ]
    }
}

with open('.harness/cd-bluegreen.yaml', 'w') as f:
    yaml.dump(bg_pipeline, f, default_flow_style=False, sort_keys=False)
print("✅ Created .harness/cd-bluegreen.yaml")
print("\n🔵🟢 Blue-Green: Stage Green → Verify → Approve → Swap Traffic")

In [ ]:
# Step 4: Compare all 3 strategies side-by-side

strategies = [
    ("Rolling", "Simplest", "Replace pods one by one", "Standard updates", "⭐"),
    ("Canary", "Safest", "Deploy to 20%, verify, then 100%", "Critical services", "⭐⭐⭐"),
    ("Blue-Green", "Fastest rollback", "Run 2 envs, swap traffic", "Zero downtime required", "⭐⭐"),
]

print("📊 DEPLOYMENT STRATEGIES COMPARISON")
print("=" * 80)
print(f"{'Strategy':<12} {'Strength':<18} {'How It Works':<35} {'Best For':<20} {'Safety'}")
print("-" * 80)
for s in strategies:
    print(f"{s[0]:<12} {s[1]:<18} {s[2]:<35} {s[3]:<20} {s[4]}")

## 🎯 POC 2 Summary

**Files created:**
| File | Purpose |
|------|---------|
| `k8s/deployment.yaml` | K8s Deployment with health probes |
| `k8s/service.yaml` | K8s Service for networking |
| `.harness/service.yaml` | Harness Service definition |
| `.harness/env_staging.yaml` | Staging environment |
| `.harness/env_prod.yaml` | Production environment |
| `.harness/cd-rolling.yaml` | Rolling deployment pipeline |
| `.harness/cd-canary.yaml` | Canary deployment pipeline |
| `.harness/cd-bluegreen.yaml` | Blue-Green deployment pipeline |

**Key Learnings:**
1. Health probes (liveness/readiness) are CRITICAL for Harness CD
2. Service = WHAT, Environment = WHERE, Strategy = HOW
3. Always use Approval Gates before production
4. Canary is safest for production, Rolling is simplest

➡️ **Next: POC 3 — A/B Test ML Models with Feature Flags!**